In [118]:
import dlrm.data as data
import tensorflow as tf

hashing_layer = tf.keras.layers.Hashing(num_bins=2**18) 

def preprocess(features, labels):
    dense_feature_names = [f'f{i}' for i in range(13)]
    sparse_feature_names = [f'c{i}' for i in range(26)]
    dense = tf.stack([features.pop(name) for name in dense_feature_names], axis=1)
    dense = tf.where(tf.math.is_nan(dense), 0., dense)
    dense = tf.where(dense < 0., 0., dense)
    dense = tf.math.log1p(dense)
    sparse = tf.stack([features.pop(name) for name in sparse_feature_names], axis=1)
    sparse = hashing_layer(sparse)
    return dense, sparse, labels

# Define column names with their types for better type inference and efficiency
column_names = ['labels'] + [f'f{i}' for i in range(13)] + [f'c{i}' for i in range(26)]
column_defaults = [0] + [0.] * 13 + ['*'] * 26

ds = tf.data.experimental.make_csv_dataset(
    '/home/aobrien/kaggle_datasets/train_1m.txt', 
    field_delim='\t', 
    column_names=column_names,
    column_defaults=column_defaults,
    batch_size=5, 
    label_name='labels',
    num_parallel_reads=tf.data.AUTOTUNE,
    sloppy=True,
    shuffle_buffer_size=10000,
)

ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [119]:
for x_dense, x_sparse, labels in ds.as_numpy_iterator():
    print(x_sparse)
    break

[[176052  67174 198018 213615 104824 163159 209555    229 126044  77865
  240385  31097 184520 165272  38205 242717  65615 214709 228323 242018
  181318 163159  24078  57738  36885  91356]
 [176052 205632 165579 237754 104824  83356 130732 195499 126044 256680
  112893 188853  61520 142865 157863 156381  65615 247603 163159 163159
  203868 163159  33334 209162 163159 163159]
 [212277  12674 148581 163787 104824  99890 221707    229 126044 188378
  213893  64751  40705 150905 181604  46927  65615  90676 228323 242018
  128333 163159  33334 219146  36885 159795]
 [176052 229790 175435 145801 172269 113408 209466 113843 126044 131567
   69873 198202 132605 142865 173993 225937  65615 184135 228323 242018
  131365 163159  66261  80309  46993  59976]
 [ 24431 194273 206280  17372 104824  99890 231002    229 126044  37685
  244010 152931 108828 124566 186203 108974  65615 114588 228323 242018
  251293   9091  33334 223495 200161 208511]]


In [122]:
import pandas as pd

df = pd.read_csv('/home/aobrien/kaggle_datasets/train_1m.txt', sep='\t', header=None)

In [123]:
df.head()

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,39
0,0,1.0,1,5.0,0.0,1382.0,4.0,15.0,2.0,181.0,...,e5ba7672,f54016b9,21ddcdc9,b1252a9d,07b5194c,NaN,3a171ecb,c5c50484,e8b83407,9727dd16
1,0,2.0,0,44.0,1.0,102.0,8.0,2.0,2.0,4.0,...,07c540c4,b04e4670,21ddcdc9,5840adea,60f6221e,NaN,3a171ecb,43f13e8b,e8b83407,731c3655
2,0,2.0,0,1.0,14.0,767.0,89.0,4.0,2.0,245.0,...,8efede7f,3412118d,NaN,NaN,e587c466,ad3062eb,3a171ecb,3b183c5c,NaN,NaN
3,0,NaN,893,NaN,NaN,4392.0,NaN,0.0,0.0,0.0,...,1e88c74f,74ef3502,NaN,NaN,6b3a5ca6,NaN,3a171ecb,9117a34a,NaN,NaN
4,0,3.0,-1,NaN,0.0,2.0,0.0,3.0,0.0,0.0,...,1e88c74f,26b3c7a7,NaN,NaN,21c9516a,NaN,32c7478e,b34f3128,NaN,NaN


In [126]:
df_sparse = df.iloc[:, 14:].fillna('*')

In [127]:
df_sparse.head()


,14,15,16,17,18,19,20,21,22,23,...,30,31,32,33,34,35,36,37,38,39
0,68fd1e64,80e26c9b,fb936136,7b4723c4,25c83c98,7e0ccccf,de7995b8,1f89b562,a73ee510,a8cd5504,...,e5ba7672,f54016b9,21ddcdc9,b1252a9d,07b5194c,*,3a171ecb,c5c50484,e8b83407,9727dd16
1,68fd1e64,f0cf0024,6f67f7e5,41274cd7,25c83c98,fe6b92e5,922afcc0,0b153874,a73ee510,2b53e5fb,...,07c540c4,b04e4670,21ddcdc9,5840adea,60f6221e,*,3a171ecb,43f13e8b,e8b83407,731c3655
2,287e684f,0a519c5c,02cf9876,c18be181,25c83c98,7e0ccccf,c78204a1,0b153874,a73ee510,3b08e48b,...,8efede7f,3412118d,*,*,e587c466,ad3062eb,3a171ecb,3b183c5c,*,*
3,68fd1e64,2c16a946,a9a87e68,2e17d6f6,25c83c98,fe6b92e5,2e8a689b,0b153874,a73ee510,efea433b,...,1e88c74f,74ef3502,*,*,6b3a5ca6,*,3a171ecb,9117a34a,*,*
4,8cf07265,ae46a29d,c81688bb,f922efad,25c83c98,13718bbd,ad9fa255,0b153874,a73ee510,5282c137,...,1e88c74f,26b3c7a7,*,*,21c9516a,*,32c7478e,b34f3128,*,*


In [130]:
vocab_sizes = df_sparse.nunique().tolist()

In [131]:
sum(vocab_sizes)

1296709

In [132]:
vocab_sizes

[1261,
 531,
 321439,
 120965,
 267,
 16,
 10863,
 563,
 3,
 30792,
 4731,
 268488,
 3068,
 26,
 8934,
 205924,
 10,
 3881,
 1855,
 4,
 240748,
 16,
 15,
 41283,
 70,
 30956]

In [ ]:
x_sparse = x_sparse.fillna('*')

In [ ]:
x_sparse

In [ ]:
import mmh3
import functools

In [ ]:
mmh3.hash.__doc__

In [ ]:
x_sparse = x_sparse.map(functools.partial(mmh3.hash, signed=False)).to_numpy(dtype='uint32')

In [ ]:
x_sparse.dtype

In [ ]:
x_sparse

In [134]:
import jax_metrics
import jax.numpy as jnp

In [153]:
import jax.numpy as jnp
import jax_metrics

# original probabilities
preds = jnp.array([0.3, 0.6, 0.2])
labels = jnp.array([0, 1, 1])

# convert to 2D (batch, num_classes)
preds_2d = jnp.stack([1 - preds, preds], axis=1)  # shape (3,2)

acc = jax_metrics.metrics.Accuracy(multiclass=True)
acc = acc.update(preds=preds_2d, target=labels)
print(acc.compute())

0.6666667


IndexError: Too many indices: 1-dimensional array indexed with 2 regular indices.